"Utilizar o dataset Multi-usuário WiMANS e filtrar por usuário. Logo após, realizar o treinamento / predição utilizando unsupervised learning. Desta forma, é possível separar as atividades por clustering ou outras técnicas de separação."

"Não supervisionado" significa que o algoritmo não sabe o que a pessoa estava fazendo (se estava andando, sentada, caindo). O trabalho dele é olhar para os números e dizer: "Olha, os dados nestes momentos são muito parecidos, vou agrupá-los no Grupo A. Estes outros são diferentes, vão para o Grupo B". Isso é o Clustering (Agrupamento).

Passo 1: processamento dos dados provenientes do WiMANS.

In [ ]:
pip install pandas numpy scipy matplotlib scikit-learn

In [ ]:
import scipy.io as sio

caminho_arquivo = '../data/ACT_100_1/act_100_1.mat'
dados_mat = sio.loadmat(caminho_arquivo)

# Descobrindo o que vem dentro do arquivo .mat
print(dados_mat.keys())


In [ ]:
# Apenas a variável trace é de fato significante
print(dados_mat['trace'])

In [ ]:
# Como funcionam os sinais na placa Intel 5300 do WiMANS?
# - Múltiplos Caminhos (MIMO): O roteador e o receptor possuem, cada um, 3 antenas (3 transmissoras e 3 receptoras).
# Como o sinal emitido por cada antena transmissora é captado por todas as receptoras, formam-se 9 caminhos 
# espaciais simultâneos 

# - Múltiplas Frequências (OFDM): Em vez de enviar uma única onda simples, a tecnologia Wi-Fi divide o canal em múltiplas 
# pequenas frequências conhecidas como subportadoras (subcarriers). A placa Intel 5300 registra o estado de 30 subportadoras
# para cada sinal emitido.

# - Ao chegarem no receptor, é calculada a amplitude de cada frequência, além da fase.

# - Portanto, a cada pacote de dados recebido (o que pode ocorrer até milhares de vezes por segundo), 
# a antena extrai a informação detalhada de 270 sub-ondas independentes (9 caminhos espaciais x 30 subportadoras)

# Através do "dtype" encontramos uma série de informações importantes:
# - timestamp_low: relogio interno do roteador (utilizado para sincronizar com o vídeo no futuro)
# - Nrx e Ntx: número de antenas transmissoras e receptoras. 
# - rssi: força do sinal. 
# - csi: número complexo que possui as informações de amplitude e fase da onda.

# IMPORTANTE: NECESSÁRIO VERIFICAR SE É POSSÍVEL JÁ OBTER O SINAL FILTRADO POR PESSOA
# Se sim, desconsiderar essa parte e avançar.
# Caso não, fazer esse filtro.

In [ ]:
import numpy as np
import pandas as pd

# 1. Pegamos a lista principal de pacotes capturados
# O '[0]' destrava a primeira barreira do formato do MATLAB
pacotes = dados_mat['trace'][0]

amplitudes_lista = []

# 2. Vamos fazer um "loop" e ler pacote por pacote (cada fração de segundo do sinal)
for i in range(len(pacotes)):
    try:
        # Acessa a matriz 'csi' dentro do pacote atual. 
        # O [0][0] quebra o aninhamento excessivo que o MATLAB cria
        csi_complexo = pacotes[i]['csi'][0][0]
        
        # 3. Matemática Mágica! Vamos tirar a Amplitude do número complexo
        # A biblioteca NumPy (np) faz isso automaticamente com o comando 'abs' (valor absoluto)
        csi_amplitude = np.abs(csi_complexo)
        
        # 4. Essa amplitude vem em formato de "Cubo" (ex: 1x3x30 subportadoras)
        # O Machine Learning prefere uma linha reta. O comando 'flatten' esmaga o cubo
        # transformando-o num barbante comprido (uma linha de dados numéricos lineares).
        csi_linha_reta = csi_amplitude.flatten()
        
        amplitudes_lista.append(csi_linha_reta)
    except IndexError:
        # Alguns pacotes podem vir corrompidos do roteador. Ignoramos eles.
        pass

# 5. Transformamos nossa lista de barbantes num formato Tabela (DataFrame) do Pandas
df_csi = pd.DataFrame(amplitudes_lista)

# Mostrando o resultado
print("A tabela ficou pronta!")
print(f"Número de amostras de tempo (linhas): {df_csi.shape[0]}")
print(f"Número de características (colunas): {df_csi.shape[1]}")

print("\nAs 5 primeiras linhas da sua Tabela de Machine Learning:")
print(df_csi.head())

A tabela ficou pronta!
Número de amostras de tempo (linhas): 1
Número de características (colunas): 270

As 5 primeiras linhas da sua Tabela de Machine Learning:
         0          1        2          3         4          5         6    \
0  21.540659  50.209561  58.5235  55.362442  52.40229  32.572995  9.055385   

         7          8          9    ...         260    261        262  \
0  22.803509  53.413481  69.231496  ...  102.044108  101.0  95.462034   

         263   264        265        266        267        268        269  
0  94.031909  87.0  72.470684  66.483081  43.011626  30.528675  28.635642  

[1 rows x 270 columns]
